In [ ]:
# TGraph records carry attributes; convert to geometry only when needed.
def _as_topology(value):
    from topologicpy.Topology import Topology
    from topologicpy.Vertex import Vertex
    if isinstance(value, (list, tuple)):
        return [_as_topology(item) for item in value]
    if not isinstance(value, dict) or "index" not in value or "dictionary" not in value:
        return value
    representation = value.get("representation")
    attributes = value["dictionary"]
    if representation is None and "src" not in value:
        representation = Vertex.ByCoordinates(
            attributes.get("x", 0), attributes.get("y", 0), attributes.get("z", 0))
    if representation is None:
        raise ValueError("This graph record has no geometry representation")
    return Topology.SetDictionary(representation, attributes)


def _record_dictionary(value):
    from topologicpy.Topology import Topology
    if isinstance(value, dict) and "dictionary" in value:
        return value["dictionary"]
    return Topology.Dictionary(value)


In [ ]:
# Import TopologicPy modules
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.TGraph import TGraph
from topologicpy.Dictionary import Dictionary
from topologicpy.Color import Color

In [ ]:
squares = []
for i in range(0,8):
    for j in range(0,8):
        v = Vertex.ByCoordinates(_as_topology(i), _as_topology(j), 0)
        sq = Face.Rectangle(origin=v)
        if i%2 == j%2:
            color = "#EEEEEE"
        else:
            color = "#000000"
        d = Dictionary.ByKeyValue("color", color)
        sq = Topology.SetDictionary(sq, d)
        squares.append(sq)

Topology.Show(squares, faceColorKey="color", faceOpacity=1)


In [ ]:
shell = Shell.ByFaces(squares)
selectors = []
for sq in squares:
    v = Topology.Centroid(_as_topology(sq))
    v = Topology.SetDictionary(v, _record_dictionary(sq))
    selectors.append(v)

shell = Topology.TransferDictionariesBySelectors(shell, selectors, tranFaces=True)

g = TGraph.ByTopology(shell)
vertices = TGraph.Vertices(g)
for v in vertices:
    d = _record_dictionary(v)
    print(Dictionary.Keys(d), Dictionary.Values(d))

In [ ]:
# Import TopologicPy modules
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

from topologicpy.Helper import Helper
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Topology import Topology
from topologicpy.Face import Face
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Wire import Wire
from topologicpy.TGraph import TGraph #importing the Graph function from Topologic
from topologicpy.Dictionary import Dictionary
import ifcopenshell
import ifcopenshell.geom
import ifcopenshell

print(Helper.Version())

file_path = r"C:\Users\sarwj\Downloads\IFC2x3_Duplex_Difference_1.ifc"

ifc_file = ifcopenshell.open(file_path)

spaces = Topology.ByIFCFile( #Topology.ByIFCFile is a function that creates a topology  from an IFC file - here the cells for each ifc space
            file=ifc_file,
            transferDictionaries=True,
            includeTypes=["IfcSpace"],
            excludeTypes=[]
        )
cells = [] #create an empty list to store the cells
for space in spaces:
    faces = Topology.Faces(_as_topology(space))
    cell = Cell.ByFaces(faces)
    cell = Topology.RemoveCoplanarFaces(cell)
    cells.append(cell) 

'''
notes after meeting with Wassims: algorithms:
graph by ifc 
topology by ifc 
get the graph by ifc file 
get the topology by ifc file for spaces, and walls, 
in the spaces, get the faces, and create a cell from these, 
then, link in the graph a face with the each of the walls creating a linked data structure'''



Topology.Show(cells, renderer = "browser")


#print(spaces)

#space_cells = Topology.Cells(spaces)

'''
doors = Topology.ByIFCFile(
            file=ifc_file,
            transferDictionaries=True,
            includeTypes=["IfcDoor"],
            excludeTypes=[]
        )

windows = Topology.ByIFCFile(
            file=ifc_file,
            transferDictionaries=True,
            includeTypes=["IfcWindow"],
            excludeTypes=[]
        )

'''

walls = Topology.ByIFCFile(
            file=ifc_file,
            transferDictionaries=True,
            includeTypes=["IfcWall", "IfcWallStandardCase"],
            excludeTypes=[]
        )

slabs = Topology.ByIFCFile(
            file=ifc_file,
            transferDictionaries=True,
            includeTypes=["IfcSlab"],
            excludeTypes=[]
        )    
'''
# Let's examine what we get from the IFC file
print("\nSpaces topology:")
print(f"Type: {type(spaces)}")
print(f"String representation: {spaces}")

# Check if it's a valid topology
is_topology = Topology.IsInstance(spaces, "Topology")
print(f"\nIs valid topology? {is_topology}")

# If it's a list or cluster, let's examine its contents
if hasattr(spaces, '__iter__'):
    print(f"\nNumber of items: {len(spaces)}")
    if len(spaces) > 0:
        print(f"First item type: {type(spaces[0])}")
        print(f"First item is topology? {Topology.IsInstance(spaces[0], 'Topology')}")
        
        # Try to get cells from the first item
        first_space_cells = Topology.Cells(spaces[0])
        if first_space_cells:
            print(f"\nNumber of cells in first space: {len(first_space_cells)}")
'''

# Print information about the spaces and cells
#print(f"\nSpaces topology type: {Topology.TypeAsString(spaces)}")

'''
# If we have cells, let's examine the first one
if len(space_cells) > 0:
    first_cell = space_cells[0]
    print(f"\nFirst cell type: {Topology.TypeAsString(first_cell)}")
    
    # Get the dictionary of the first cell
    cell_dict = Topology.Dictionary(first_cell)
    if cell_dict:
        print("\nFirst cell properties:")
        for key in cell_dict.Keys():
            print(f"{key}: {cell_dict.ValueAtKey(key)}")
'''


#Topology.Show(spaces, renderer = "browser")
#Topology.Show(walls,slabs, renderer = "browser")



In [ ]:
def Isovist(face, vertex, obstacles: list = [], direction: list = [0,1,0], fov: float = 360, transferDictionaries: bool = False, mantissa: int = 6, tolerance: float = 0.0001):
    """
    Returns the face representing the isovist projection from the input viewpoint.
    This method assumes all input is in 2D. Z coordinates are ignored.

    Parameters
    ----------
    face : topologic_core.Face
        The face representing the boundary of the isovist.
    vertex : topologic_core.Vertex
        The vertex representing the location of the viewpoint of the isovist.
    obstacles : list , optional
        A list of wires representing the obstacles within the face. All obstacles are assumed to be within the
        boundary of the face. The default is [].
    direction : list, optional
        The vector representing the direction (in the XY plane) in which the observer is facing. The Z component is ignored.
        The direction follows the Vector.CompassAngle convention where [0,1,0] (North) is considered to be
        in the positive Y direction, [1,0,0] (East) is considered to be in the positive X-direction.
        Angles are measured in a clockwise fashion. The default is [0,1,0] (North).
    fov : float , optional
        The horizontal field of view (fov) angle in degrees. See https://en.wikipedia.org/wiki/Field_of_view.
        The acceptable range is 1 to 360. The default is 360.
    transferDictionaries : bool , optional
        If set to True, the dictionaries of the encountered edges will be transfered to the isovist edges. The default is False.
    mantissa : int , optional
        The desired length of the mantissa. The default is 6.
    tolerance : float , optional:
        The desired tolerance. The default is 0.0001.

    Returns
    -------
    topologic_core.Face
        The face representing the isovist projection from the input viewpoint.

    """
    from topologicpy.Vertex import Vertex
    from topologicpy.Edge import Edge
    from topologicpy.Wire import Wire
    from topologicpy.Face import Face
    from topologicpy.Shell import Shell
    from topologicpy.Cluster import Cluster
    from topologicpy.Topology import Topology
    from topologicpy.Vector import Vector

    def vertexPartofFace(vertex, face, tolerance):
        vertices = Topology.Vertices(_as_topology(face))
        for v in vertices:
            if Vertex.Distance(_as_topology(vertex), _as_topology(v)) < tolerance:
                return True
        return False

    if not Topology.IsInstance(_as_topology(face), 'Face'):
        print("Face.Isovist - Error: The input boundary parameter is not a valid Face. Returning None")
        return None
    if not Topology.IsInstance(_as_topology(vertex), 'Vertex'):
        print("Face.Isovist - Error: The input viewPoint parameter is not a valid Vertex. Returning None")
        return None
    if fov < 1 or fov > 360:
        print("Face.Isovist - Error: The input fov parameter is outside the acceptable range of 0 to 360 degrees. Returning None")
        return None
    if isinstance(obstacles, list):
        obstacles = [obs for obs in obstacles if Topology.IsInstance(_as_topology(obs), 'Wire')]
    else:
        obstacles = []
    
    origin = Topology.Centroid(_as_topology(face))
    normal = Face.Normal(face)
    flat_face = Topology.Flatten(face, origin=origin, direction=normal)
    flat_vertex = Topology.Flatten(vertex, origin=origin, direction=normal)
    flat_obstacles = [Topology.Flatten(obstacle, origin=origin, direction=normal) for obstacle in obstacles]

    eb = Face.ExternalBoundary(flat_face)
    vertices = Topology.Vertices(_as_topology(eb))
    coords = [Vertex.Coordinates(_as_topology(v), outputType='xy') for v in vertices]
    new_vertices = [Vertex.ByCoordinates(_as_topology(coord)) for coord in coords]
    eb = Wire.ByVertices(_as_topology(new_vertices), close=True)

    ib_list = Face.InternalBoundaries(flat_face)
    new_ib_list = []
    for ib in ib_list:
        vertices = Topology.Vertices(_as_topology(ib))
        coords = [Vertex.Coordinates(_as_topology(v), outputType='xy') for v in vertices]
        new_vertices = [Vertex.ByCoordinates(_as_topology(coord)) for coord in coords]
        new_ib_list.append(Wire.ByVertices(_as_topology(new_vertices), close=True))

    flat_face = Face.ByWires(eb, new_ib_list)
    for obs in flat_obstacles:
        flat_face = Topology.Difference(flat_face, Face.ByWire(obs))
    
    targets = Topology.Vertices(_as_topology(flat_face))
    distances = []
    for target in targets:
        distances.append(Vertex.Distance(_as_topology(flat_vertex), _as_topology(target)))
    distances.sort()
    max_d = distances[-1]*1.05
    edges = []
    for target in targets:
        if Vertex.Distance(_as_topology(flat_vertex), _as_topology(target)) > tolerance:
            e = Edge.ByVertices(_as_topology(flat_vertex), _as_topology(target), silent=True)
            e = Edge.SetLength(_as_topology(e), length=max_d, bothSides=False, tolerance=tolerance)
            edges.append(e)
    shell = Topology.Slice(flat_face, Cluster.ByTopologies(_as_topology(edges)))
    faces = Topology.Faces(_as_topology(shell))
    final_faces = []
    for f in faces:
        if vertexPartofFace(flat_vertex, f, tolerance=0.001):
            final_faces.append(f)
    shell = Shell.ByFaces(final_faces)
    return_face = Topology.RemoveCoplanarFaces(shell, epsilon=0.1)

    if not Topology.IsInstance(_as_topology(return_face), 'face'):
        temp = Shell.ExternalBoundary(shell)
        if Topology.IsInstance(_as_topology(temp), 'Wire'):
            return_face = Face.ByWire(temp)
        elif Topology.IsInstance(_as_topology(temp), 'Cluster'):
            edges = Topology.Edges(_as_topology(temp))
            vertices = Topology.Vertices(_as_topology(temp))
            vertices = Vertex.Fuse(_as_topology(vertices), tolerance=0.01)
            new_edges = []
            for edge in edges:
                sv = vertices[Vertex.Index(_as_topology(Edge.StartVertex(_as_topology(edge))), _as_topology(vertices), tolerance=0.01)]
                ev = vertices[Vertex.Index(_as_topology(Edge.EndVertex(_as_topology(edge))), _as_topology(vertices), tolerance=0.01)]
                if Vertex.Distance(_as_topology(sv), _as_topology(ev)) > tolerance:
                    new_edges.append(Edge.ByVertices(_as_topology([sv, ev])))
            w = Wire.ByEdges(_as_topology(new_edges), tolerance=0.01)
            return_face = Face.ByWire(w)
        if not Topology.IsInstance(_as_topology(return_face), 'Face'):
            print("Face.Isovist - Error: Could not create isovist. Returning None.")
            return None 
    compAngle = 0
    if fov == 360:
        pie = Face.Circle(origin= flat_vertex, radius=max_d*1.2, sides=16)
    else:
        compAngle = Vector.CompassAngle(Vector.North(), direction, mantissa=mantissa, tolerance=tolerance)
        fromAngle =  compAngle - fov*0.5 
        toAngle = compAngle + fov*0.5
        c = Wire.Circle(origin= flat_vertex, radius=max_d*1.2, sides=int(16*(fov/360)), fromAngle=fromAngle, toAngle=toAngle, close = False)
        e1 = Edge.ByVertices(_as_topology(Wire.StartVertex(_as_topology(c))), _as_topology(flat_vertex), silent=True)
        e2 = Edge.ByVertices(_as_topology(Wire.EndVertex(_as_topology(c))), _as_topology(flat_vertex), silent=True)
        edges = Topology.Edges(_as_topology(c)) + [e1,e2]
        pie = Face.ByWire(Topology.SelfMerge(Cluster.ByTopologies(_as_topology(edges))))
    return_face = Topology.Intersect(pie, return_face)
    if not Topology.IsInstance(_as_topology(return_face), 'face'):
        return_face = Topology.SelfMerge(return_face)
    if return_face == None:
        print("Face.Isovist - Error: Could not create isovist. Returning None.")
        return None
    simpler_face = Face.RemoveCollinearEdges(return_face)
    if Topology.IsInstance(_as_topology(simpler_face), 'face'):
        if transferDictionaries == True:
            j_edges = [Topology.Edges(_as_topology(t)) for t in obstacles]
            j_edges = Helper.Flatten(j_edges)
            j_edges += Topology.Edges(_as_topology(face))
            i_edges = Topology.Edges(_as_topology(simpler_face))
            used = [0 for _ in range(len(j_edges))]
            for i, i_edge in enumerate(i_edges):
                d_i = _record_dictionary(i_edge)
                for j, j_edge in enumerate(j_edges):
                    if used[j] == 0:
                        if Edge.IsCollinear(_as_topology(i_edge), _as_topology(j_edge)):
                            d_j = _record_dictionary(j_edge)
                            d_result = Dictionary.ByMergedDictionaries([d_i, d_j])
                            i_edge = Topology.SetDictionary(i_edge, d_result)
                            used[j] == 1
        return Topology.Unflatten(simpler_face, origin=origin, direction=normal)
    return Topology.Unflatten(return_face, origin=origin, direction=normal)

# Import TopologicPy modules
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

from topologicpy.Helper import Helper
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Topology import Topology
from topologicpy.Face import Face
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Wire import Wire
from topologicpy.TGraph import TGraph #importing the Graph function from Topologic
from topologicpy.Dictionary import Dictionary
from topologicpy.Color import Color

colors = Color.CSSNamedColors()
counter = 0
r = Face.Rectangle(width=10, length=10)
edges = Topology.Edges(_as_topology(r))
for i, edge in enumerate(edges):
    d = Dictionary.ByKeysValues(["type","id"], ["boundary",colors[i+30]])
    edge = Topology.SetDictionary(edge, d)
    counter += 1
r1 =Wire.Rectangle(width=2, length=2)
r1 = Topology.Translate(_as_topology(r1), x=3, y=2.5, z=0)
r2 =Wire.Rectangle(width=2, length=2)
r2 = Topology.Translate(_as_topology(r2), x=-2, y=-3, z=0)
edges = Topology.Edges(_as_topology(r1))
for i, edge in enumerate(edges):
    d = Dictionary.ByKeysValues(["type", "id"], ["retail","red"])
    edge = Topology.SetDictionary(edge, d)
    counter += 1

edges = Topology.Edges(_as_topology(r2))
for i, edge in enumerate(edges):
    d = Dictionary.ByKeysValues(["type","id"], ["residential","yellow"])
    edge = Topology.SetDictionary(edge, d)
    counter += 1
v = Vertex.ByCoordinates(-3,0,0)
d = Dictionary.ByKeyValue("vertexSize", 10)
v = Topology.SetDictionary(v, d)
isovist = Face.Isovist(face=r, obstacles=[r1, r2], vertex=v, fov=140, direction=[1,-1,0], transferDictionaries=True, metrics=True)
#isovist = Topology.Move(isovist, 12, 0, 0)
edges = Topology.Edges(_as_topology(isovist))
for edge in edges:
    d = _record_dictionary(edge)
    d = Dictionary.SetValueAtKey(d, "edgeWidth", 5)
    edge = Topology.SetDictionary(edge, d)
isovist2 = Topology.Translate(_as_topology(isovist), 12, 0, 0)

v2 = Topology.Translate(_as_topology(v), 12, 0, 0)

r_b = Topology.Move(r, -12, 0, 0)
r1_b = Topology.Move(r1, -12, 0, 0)
r2_b = Topology.Move(r2, -12, 0, 0)
v_b = Topology.Move(v, -12, 0, 0)
d = _record_dictionary(isovist)
keys = Dictionary.Keys(d)
for i, key in enumerate(keys):
    print(str(i+1).zfill(2)+".", key, Dictionary.ValueAtKey(d, key))

v_furthest = Vertex.ByCoordinates(_as_topology(Dictionary.ValueAtKey(d, 'v_max')))
max_edge = Edge.ByVertices(_as_topology(v), _as_topology(v_furthest))
d = _record_dictionary(isovist)
d = Dictionary.SetValueAtKey(d, "faceOpacity", 0.5)
isovist = Topology.SetDictionary(isovist, d)
Topology.Show(r, r1,r2,r_b,r1_b,r2_b,isovist,isovist2,v,v2, v_b, max_edge, vertexSizeKey="vertexSize", edgeColor="black",
              edgeColorKey="id", edgeWidth=4, edgeWidthKey="edgeWidth", faceOpacityKey="faceOpacity", showFaces=True, backgroundColor="grey")

In [ ]:

edges = Topology.Edges(_as_topology(isovist))
dictionaries = [_record_dictionary(edge) for edge in edges]
temp_dict = Helper.ClusterByKeys(edges, dictionaries, "type", silent=True)
dict_clusters = temp_dict["dictionaries"]
elements_clusters = temp_dict['elements']
for j, elements_cluster in enumerate(elements_clusters):
    total_length = sum([Edge.Length(_as_topology(e)) for e in elements_cluster])
    d = dict_clusters[j][0] # All dicitonaries have same values in dictionaries, so take first one.
    print("Type:", Dictionary.ValueAtKey(d, "type") or "None", ",    Total Length:", total_length)